# Experiment: Case-001 country registry sweep gate

Objective: convert the July 11 country-specific ClinicalTrials.gov sweep into public source-radar labels only, without producing eligibility, referral, trial-screening, or treatment outputs.

In [ ]:
import json
from collections import Counter
from pathlib import Path

repo = Path.cwd()
snapshot_path = (
    repo
    / "data/registries/clinicaltrials"
    / "2026-07-11-country-active-beta-thalassemia-sweep.json"
)
snapshot = json.loads(snapshot_path.read_text(encoding="utf-8"))

assert snapshot["checked_date"] == "2026-07-11"
assert snapshot["condition_query"] == "beta thalassemia"
len(snapshot["countries"])

## Plan

- Count country-specific hits and zero-query countries.
- Deduplicate registry IDs across countries.
- Preserve status and latest-update signals.
- Emit source-radar and owner-verification labels only.

In [ ]:
countries_with_records = [
    country["country_query"]
    for country in snapshot["countries"]
    if country["total_count"]
]
zero_countries = [
    country["country_query"]
    for country in snapshot["countries"]
    if not country["total_count"]
]

assert countries_with_records == [
    "China",
    "India",
    "Japan",
    "Korea",
    "Malaysia",
    "Thailand",
    "Turkey",
    "Pakistan",
    "Saudi Arabia",
    "Taiwan",
]
assert zero_countries == [
    "Hong Kong",
    "Indonesia",
    "Singapore",
    "Iran",
    "Russia",
]
(countries_with_records, zero_countries)

In [ ]:
def unique_records_by_nct() -> dict[str, dict]:
    """Deduplicate records that appear in multi-country studies."""
    unique: dict[str, dict] = {}
    for country in snapshot["countries"]:
        for record in country["records"]:
            unique[record["nct_id"]] = record
    return unique


unique_records = unique_records_by_nct()
status_counts = Counter(record["status"] for record in unique_records.values())

assert len(unique_records) == 30
assert status_counts == {
    "RECRUITING": 21,
    "ENROLLING_BY_INVITATION": 4,
    "NOT_YET_RECRUITING": 5,
}
dict(status_counts)

In [ ]:
latest_updates = sorted(
    (
        record["last_update"],
        nct_id,
        record["status"],
        record["title"],
    )
    for nct_id, record in unique_records.items()
    if record["last_update"]
)
latest_updates = list(reversed(latest_updates))[:8]

required_ids = {"NCT07673302", "NCT07215975", "NCT05477563"}
assert required_ids <= set(unique_records)
assert latest_updates[0][1] == "NCT04143724"
latest_updates

In [ ]:
allowed_outputs = {
    "country_registry_sweep_ready_benchmark_only",
    "country_specific_registry_query_required",
    "owner_verification_required",
    "branch_review_handoff_packet_incomplete",
}
blocked_outputs = {
    "diagnosis",
    "eligibility",
    "trial_screening",
    "referral",
    "travel",
    "import",
    "purchase",
    "dosing",
    "treatment_selection",
    "sample_routing",
}
assert not (allowed_outputs & blocked_outputs)

summary = {
    "label": "country_registry_sweep_ready_benchmark_only",
    "country_queries": len(snapshot["countries"]),
    "countries_with_records": countries_with_records,
    "zero_countries": zero_countries,
    "unique_nct_count": len(unique_records),
    "status_counts": dict(status_counts),
    "case001_state": "branch_review_handoff_packet_incomplete",
}
summary

## Result

- The country-specific sweep recovered 30 unique active-status registry IDs across 10 country queries.
- Five country queries returned zero public ClinicalTrials.gov records in this pass: Hong Kong, Indonesia, Singapore, Iran, and Russia.
- The zero-country result is a registry-query label only; it does not prove no clinical access, hospital program, local registry, or private pathway exists.
- The only allowed next action is source disambiguation and qualified owner verification.